<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/SK_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Clone Sebastian Raschka's repository
!git clone https://github.com/rasbt/LLMs-from-scratch.git
%cd LLMs-from-scratch

# 2. Install requirements
!pip install -r requirements.txt

In [7]:
# ============================================================================
# SELF-CONTAINED GOVERNED GPT PIPELINE (RASCHKA ARCHITECTURE + TOPO GOVERNOR)
# SEED: 123 | TIERS 0 - 3 ACTIVE MANIFOLD PERMANENCE
# ============================================================================

import os
import random
import math
import time
import hashlib
import urllib.request
import zipfile
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================================
# 1. Deterministic Seeding Protocol (Seed 123)
# ============================================================================

def set_seed(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(123)

# ============================================================================
# 2. Configuration & Invariant Constants
# ============================================================================

@dataclass
class TopoConfig:
    prime_anchors: List[int] = field(default_factory=lambda: [2, 3, 5, 7, 11, 13])
    safety_constant: float = 0.9785142874
    sigma_critical: float = 0.5
    epsilon_geodesic: float = 1e-9
    prime_to_equity: Dict[int, str] = field(default_factory=lambda: {
        2: "Parity Invariance",
        3: "Ternary Equilibrium",
        5: "Pentagonal Symmetry",
        7: "Heptagonal Stability",
        11: "Subspace Orthogonality",
        13: "Manifold Permanence"
    })

config = TopoConfig()

# ============================================================================
# 3. TIER 0: Data-Spectral Integrity Layer
# ============================================================================

class DataSpectralIntegrityLayer:
    def __init__(self, reference_set: List[int] = None, threshold: float = None):
        self.reference_set = reference_set or config.prime_anchors
        self.threshold = threshold or config.safety_constant
        self.rejected_samples = []
        self.passed_samples = []
        self.reference_tensor = torch.tensor(self.reference_set, dtype=torch.float64)
        self._reference_norm = torch.norm(self.reference_tensor)

    def compute_spectral_signature(self, sample: torch.Tensor) -> torch.Tensor:
        sample = sample.float()
        signature = torch.zeros(len(self.reference_set), device=sample.device)
        flat_prefix = sample.flatten()[:100]
        prefix_mean = torch.mean(flat_prefix) if flat_prefix.numel() > 0 else torch.tensor(0.0, device=sample.device)

        for i, prime in enumerate(self.reference_set):
            projection = prefix_mean * prime
            signature[i] = projection / (prime + 1)

        norm = torch.norm(signature)
        if norm > 0:
            signature = signature / norm
        return signature

    def compute_distance(self, signature: torch.Tensor) -> float:
        ref_normalized = (self.reference_tensor / self._reference_norm).to(signature.device)
        return torch.norm(signature.to(torch.float64) - ref_normalized).item()

    def detect_bias(self, sample: torch.Tensor) -> Dict:
        signature = self.compute_spectral_signature(sample)
        distance = self.compute_distance(signature)
        bias_score = min(1.0, distance / (1.0 - self.threshold + 1e-9))
        status = "PURE" if distance <= (1.0 - self.threshold) else "BIASED"
        return {'signature': signature, 'distance': distance, 'bias_score': bias_score, 'status': status}

    def process_batch(self, samples: torch.Tensor) -> Tuple[torch.Tensor, Dict]:
        if len(samples.shape) == 1:
            samples = samples.unsqueeze(0)
        filtered = []
        rejected_info = []
        for i in range(samples.shape[0]):
            result = self.detect_bias(samples[i])
            if result['status'] == "BIASED":
                self.rejected_samples.append(result)
                rejected_info.append({'index': i, 'bias_score': result['bias_score']})
            else:
                filtered.append(samples[i])
                self.passed_samples.append(result)
        return (torch.stack(filtered) if filtered else torch.tensor([], device=samples.device)), {
            'total_processed': samples.shape[0],
            'rejected_count': len(rejected_info),
            'passed_count': len(filtered),
            'rejection_rate': len(rejected_info) / max(1, samples.shape[0])
        }

    def get_audit_report(self) -> Dict:
        total = len(self.rejected_samples) + len(self.passed_samples)
        return {
            'total_processed': total,
            'rejected_count': len(self.rejected_samples),
            'passed_count': len(self.passed_samples),
            'rejection_rate': len(self.rejected_samples) / max(1, total)
        }

# ============================================================================
# 4. TIER 1: L-EFM Operator
# ============================================================================

class LEFMOperator:
    def __init__(self, sigma: float = 0.5):
        self.sigma = sigma

    def compute_spectral_trap(self, sigma: float) -> float:
        if abs(sigma - 0.5) < 1e-6:
            return 1.0
        return math.exp(-((sigma - 0.5) ** 2) * 50)

    def annihilate_bias(self, spectral_vector: torch.Tensor) -> torch.Tensor:
        result = torch.zeros_like(spectral_vector)
        for i in range(len(spectral_vector)):
            trap_value = self.compute_spectral_trap(float(torch.abs(spectral_vector[i]).item()))
            if abs(trap_value - 1.0) < 1e-6:
                result[i] = spectral_vector[i]
        return result

    def verify_purity(self, vector: torch.Tensor) -> Tuple[bool, float]:
        if vector.numel() == 0:
            return False, 0.0
        trap_values = []
        for i in range(min(len(vector), 100)):
            trap_values.append(self.compute_spectral_trap(float(torch.abs(vector[i]).item())))
        purity = sum(1 for v in trap_values if abs(v - 1.0) < 1e-6) / len(trap_values)
        return purity > 0.95, purity

# ============================================================================
# 5. TIER 2: H2E-Sheriff-BIAS
# ============================================================================

class H2ESheriffBIAS:
    def __init__(self, epsilon: float = 1e-9):
        self.epsilon = epsilon
        self.equitable_geodesic = torch.tensor([0.0, 0.0])
        self.violations = []

    def _compute_hyperbolic_distance(self, p1: torch.Tensor, p2: torch.Tensor) -> float:
        p_norm = torch.norm(p1).item()
        q_norm = torch.norm(p2).item()
        if p_norm >= 1.0 or q_norm >= 1.0:
            return float('inf')
        numerator = 2 * torch.norm(p1 - p2).item() ** 2
        denominator = (1 - p_norm ** 2) * (1 - q_norm ** 2)
        if denominator <= 0:
            return float('inf')
        cosh_dist = 1 + numerator / denominator
        if cosh_dist < 1:
            return 0.0
        return math.acosh(cosh_dist)

    def verify_constructible(self, tensor: torch.Tensor) -> Tuple[bool, float, Dict]:
        device = tensor.device
        hyperbolic = tensor[:2] if tensor.numel() >= 2 else torch.zeros(2, device=device)
        distance = self._compute_hyperbolic_distance(hyperbolic.float().cpu(), self.equitable_geodesic)
        is_cons = distance <= self.epsilon
        if not is_cons:
            self.violations.append({'distance': distance, 'timestamp': time.time()})
        return is_cons, distance, {'distance': distance, 'constructible': is_cons}

# ============================================================================
# 6. TIER 3: Prime-Anchored Equity Governor
# ============================================================================

class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = 13):
        self.embed_layer = embed_layer
        self.tier0 = DataSpectralIntegrityLayer()
        self.tier1 = LEFMOperator()
        self.tier2 = H2ESheriffBIAS()

        vocab_size = embed_layer.weight.shape[0]
        sieve = [True] * (prime_limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(prime_limit ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, prime_limit + 1, i):
                    sieve[j] = False
        primes = [i for i in range(2, prime_limit + 1) if sieve[i]]
        self.anchor_indices = [p for p in primes if p < vocab_size]
        self.snapshot = {}
        self.bias_rejections = 0
        self.total_processed = 0
        self.spectral_traps_triggered = 0
        self.geometric_violations = 0

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        device = self.embed_layer.weight.device
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(device=device, dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        device = self.embed_layer.weight.device
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached.to(device), atol=atol)
            for idx, cached in self.snapshot.items()
        )

    def get_hash(self) -> str:
        hasher = hashlib.sha256()
        for idx in self.anchor_indices:
            weight_bytes = self.embed_layer.weight[idx].detach().float().cpu().numpy().tobytes()
            hasher.update(weight_bytes)
        return hasher.hexdigest()[:16]

    def get_equity_anchors(self) -> Dict[int, str]:
        return {p: config.prime_to_equity[p] for p in self.anchor_indices if p in config.prime_to_equity}

    def get_anchor_memory_kb(self) -> float:
        return (len(self.anchor_indices) * self.embed_layer.weight.shape[1] * 4) / 1024

    def process_data(self, sample: torch.Tensor) -> Dict:
        self.total_processed += 1
        tier0_result = self.tier0.detect_bias(sample)
        if tier0_result['status'] == "BIASED":
            self.bias_rejections += 1
            return {'passed': False, 'tier': 0}

        annihilated = self.tier1.annihilate_bias(tier0_result['signature'])
        is_pure, purity = self.tier1.verify_purity(annihilated)
        if not is_pure:
            self.spectral_traps_triggered += 1
            return {'passed': False, 'tier': 1}

        is_cons, dist, info = self.tier2.verify_constructible(annihilated)
        if not is_cons:
            self.geometric_violations += 1
            return {'passed': False, 'tier': 2}

        return {'passed': True}

    def get_audit_report(self) -> Dict:
        return {
            'total_processed': self.total_processed,
            'bias_rejections': self.bias_rejections,
            'spectral_traps_triggered': self.spectral_traps_triggered,
            'geometric_violations': self.geometric_violations,
            'rejection_rate': self.bias_rejections / max(1, self.total_processed),
            'anchor_hash': self.get_hash(),
            'anchor_memory_kb': self.get_anchor_memory_kb()
        }

# ============================================================================
# 7. Self-Contained GPT Transformer Backbone (Raschka Architecture)
# ============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(-2, -1)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            nn.GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )
    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        return x + shortcut

class GovernedGPTClassifier(nn.Module):
    def __init__(self, cfg, num_classes=2, prime_limit=13):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], num_classes)

        # Attach Tier 3 Governor directly to token embeddings
        self.governor = TopologicalGovernor(self.tok_emb, prime_limit=prime_limit)

    def forward(self, in_idx):
        b, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        last_token = x[:, -1, :]
        return self.out_head(last_token)

# ============================================================================
# 8. Raschka Dataset Downloader & Simple Byte-Pair Tokenizer
# ============================================================================

def get_spam_dataloader(batch_size=8, max_length=120):
    url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
    zip_path = "sms_spam_collection.zip"
    extracted_path = "SMSSpamCollection"

    if not Path(extracted_path).exists():
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(".")

    df = pd.read_csv(extracted_path, sep="\t", header=None, names=["label", "text"])
    df["label"] = df["label"].map({"ham": 0, "spam": 1})

    try:
        import tiktoken
        tokenizer = tiktoken.get_encoding("gpt2")
        encode_fn = lambda s: tokenizer.encode(s)
    except ImportError:
        encode_fn = lambda s: [hash(w) % 50257 for w in s.split()]

    class SpamDataset(Dataset):
        def __init__(self, data_df, max_len):
            self.texts = data_df["text"].tolist()
            self.labels = data_df["label"].tolist()
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            tokens = encode_fn(self.texts[idx])
            if len(tokens) > self.max_len:
                tokens = tokens[:self.max_len]
            else:
                tokens = tokens + [50256] * (self.max_len - len(tokens))
            return torch.tensor(tokens, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

    sample_df = pd.concat([df[df["label"] == 0].head(100), df[df["label"] == 1].head(100)]).reset_index(drop=True)
    dataset = SpamDataset(sample_df, max_len=max_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ============================================================================
# 9. Governed Continual Training Loop
# ============================================================================

def train_governed_epoch(
    model: GovernedGPTClassifier,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> Dict:
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        # TIER 0-2: Manifold & Spectral Integrity Audit
        with torch.no_grad():
            emb_repr = model.tok_emb(batch_x)
            _ = model.governor.process_data(emb_repr)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()

        # TIER 3: Zero Out Anchor Gradients (Active Surgery)
        model.governor.zero_anchor_gradients()

        optimizer.step()

        # TIER 3: Enforce Manifold Invariance
        model.governor.enforce_anchors()

        total_loss += loss.item()

    integrity_passed = model.governor.verify_integrity(atol=1e-6)
    return {
        "loss": total_loss / len(loader),
        "integrity_passed": integrity_passed,
        "audit": model.governor.get_audit_report()
    }

# ============================================================================
# 10. Execution Pipeline
# ============================================================================

if __name__ == "__main__":
    set_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INIT] Executing Raschka GPT Model + Topological Governor on: {device} | Seed: 123")

    # 4-block transformer backbone matching Raschka's modular format
    GPT_CONFIG = {
        "vocab_size": 50257,
        "context_length": 128,
        "emb_dim": 256,
        "n_heads": 4,
        "n_layers": 4,
        "drop_rate": 0.1,
        "qkv_bias": False
    }

    print("[SETUP] Instantiating Governed GPT Architecture...")
    model = GovernedGPTClassifier(GPT_CONFIG, num_classes=2, prime_limit=13).to(device)

    # Take baseline snapshot on target device
    model.governor.take_snapshot()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    print(f"[TIER 3 GOVERNOR] Active Anchor Indices: {model.governor.anchor_indices}")
    print(f"[TIER 3 GOVERNOR] Equity Anchors: {model.governor.get_equity_anchors()}")
    print(f"[TIER 3 GOVERNOR] Anchor Memory Footprint: {model.governor.get_anchor_memory_kb():.2f} KB (O(1))")
    print(f"[TIER 3 GOVERNOR] Initial Anchor Hash: {model.governor.get_hash()}")

    # Prepare real SMS Spam DataLoader
    print("[DATA] Loading SMS Spam Dataset...")
    train_loader = get_spam_dataloader(batch_size=8, max_length=128)

    # Train under governor protection
    print("\n[TRAINING] Starting Governed Continual Fine-Tuning...")
    for epoch in range(1, 4):
        stats = train_governed_epoch(model, train_loader, optimizer, device)
        print(f"Epoch {epoch} | Loss: {stats['loss']:.4f} | Manifold Integrity: {stats['integrity_passed']}")

    # Output audit report
    audit = model.governor.get_audit_report()
    print("\n[FINAL ARCHITECTURAL AUDIT REPORT]")
    for k, v in audit.items():
        print(f"  - {k}: {v}")

[INIT] Executing Raschka GPT Model + Topological Governor on: cuda | Seed: 123
[SETUP] Instantiating Governed GPT Architecture...
[TIER 3 GOVERNOR] Active Anchor Indices: [2, 3, 5, 7, 11, 13]
[TIER 3 GOVERNOR] Equity Anchors: {2: 'Parity Invariance', 3: 'Ternary Equilibrium', 5: 'Pentagonal Symmetry', 7: 'Heptagonal Stability', 11: 'Subspace Orthogonality', 13: 'Manifold Permanence'}
[TIER 3 GOVERNOR] Anchor Memory Footprint: 6.00 KB (O(1))
[TIER 3 GOVERNOR] Initial Anchor Hash: b8854b813c71e78c
[DATA] Loading SMS Spam Dataset...

[TRAINING] Starting Governed Continual Fine-Tuning...
Epoch 1 | Loss: 0.7161 | Manifold Integrity: True
Epoch 2 | Loss: 0.5437 | Manifold Integrity: True
Epoch 3 | Loss: 0.4165 | Manifold Integrity: True

[FINAL ARCHITECTURAL AUDIT REPORT]
  - total_processed: 75
  - bias_rejections: 75
  - spectral_traps_triggered: 0
  - geometric_violations: 0
  - rejection_rate: 1.0
  - anchor_hash: b8854b813c71e78c
  - anchor_memory_kb: 6.0
